# Lab: Fairness Through Group Reweighting (Solutions)

This lab studies a concrete question: when a single linear classifier is trained
with different weights for different demographic groups, how do common fairness
criteria change?

The optimizer is provided by the BADR package (`badr`). Your implementation task
is limited to three metric functions. The rest of the notebook compares the
models obtained from different group weightings.

For group $s$, let $\ell_s(w)$ denote the average regularized logistic loss on
that group. For weights $\lambda$ on the simplex,

$$
w^\star(\lambda) = \arg\min_w \sum_{s=1}^{S} \lambda_s\,\ell_s(w),
\qquad
V(\lambda) = \mathrm{metric}\big(w^\star(\lambda)\big).
$$

Thus a choice of $\lambda$ determines a fitted model and a value of the selected
fairness metric. BADR chooses $\lambda$ for the metric under consideration.

**What you will do.**
1. Implement three fairness metrics using the provided helper functions.
2. Compare several fixed weightings: uniform over groups, size-balanced,
   min-max, and single-group weightings.
3. Run BADR for one metric.
4. Use simplex plots to compare how these choices behave for different metrics.

<a id="toc"></a>
## Table of Contents
1. [Setup and Data](#sec-setup)
2. [Fairness Metrics](#sec-metrics)
3. [Fixed Group Weightings](#sec-reweight)
4. [Run BADR](#sec-badr)
5. [Simplex Plots](#sec-figure)
6. [Discussion](#sec-discussion)


> **Reference.** The `badr` documentation describes the dataset loaders,
> metrics, models, and the high-level `Badr` API:
>
> <https://badr.readthedocs.io/en/latest/>


## 1. Setup and Data <a class="anchor" id="sec-setup"></a>

The next cell installs BADR and downloads the helper module. Run it in Colab or
in a fresh environment. If you are working from a local clone that already has
`badr` and `lab_helpers.py`, skip the installation cell and run the imports.


In [ ]:
# Install BADR and download the lab helpers (for Colab or a fresh environment).
!pip install -q badr
!wget -q -O lab_helpers.py https://raw.githubusercontent.com/AdaptiveDecisionMakingGroup/badr/lab/lab_helpers.py


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp

import badr               # datasets, models, metrics, algorithms
import lab_helpers as lh  # lab utilities for data, tables, and plots


### 1.1. Dataset and Model

We use the binary **ACSEmployment** task (`badr.datasets.fetch_ACSEmployment`):
predict whether a person is employed. The data are from Hawaii in 2016. The
sex-by-race buckets are combined into **three groups**, and the features are
standardized.

The dataset stores grouped training arrays as `X_train_list[s]` and
`y_train_list[s]`. The model is `badr.models.LogisticRegression` without an
intercept, so the score for observation $x_i$ is $s_i = x_i^\top w$. For group
$s$,

$$
\ell_s(w) = \frac{1}{n_s}\sum_{i \in G_s}\big[\mathrm{softplus}(x_i^\top w) - y_i\,x_i^\top w\big] + \tfrac{\mu}{2}\lVert w\rVert^2 .
$$

The plots later use a three-group simplex, so the notebook checks that
`n_groups == 3`.


In [ ]:
dset = badr.datasets.fetch_ACSEmployment(states=["HI"], year=2016, n_groups=3)
model = badr.models.LogisticRegression(l2_reg=1e0, fit_intercept=False)

# Grouped training arrays used in the metric implementations.
X_list, y_list = lh.group_lists(dset, "train")
n_groups = dset.n_groups

assert n_groups == 3, f"expected 3 groups, got {n_groups}"
print("groups:", n_groups, "| features:", dset.n_features,
      "| group sizes:", [int(len(y)) for y in y_list])


## 2. Fairness Metrics <a class="anchor" id="sec-metrics"></a>

A metric receives fitted coefficients $w$ and returns a scalar value. Smaller
values are treated as better for the criterion being measured. Implement the
three functions below with signature

```python
metric(w, X_list, y_list)  ->  scalar
```

where the score for observation $x_i$ is $s_i = x_i^\top w$.

The next cell provides four helper functions: `soft_max`, `soft_min`,
`group_scores(w, X_list)`, and `soft_rates(w, X_list, y_list, rho)`. The last
function returns soft TPR and FPR values for each group, using
$\sigma(\rho x_i^\top w)$ instead of a hard threshold. The functions

$$
\widetilde{\max}_\rho(v) = \tfrac{1}{\rho}\log\textstyle\sum_k e^{\rho v_k},
\qquad
\widetilde{\min}_\rho(v) = -\widetilde{\max}_\rho(-v)
$$

are smooth versions of max and min.

### 2.1. Individual Fairness — weighted score gaps
$$
\mathrm{IF}(w) = \frac{1}{Z}\sum_{a<b}\sum_{i \in G_a}\sum_{j \in G_b}
\exp(-|y_i - y_j|)\,(s_i - s_j)^2, \qquad Z = \sum_{a \ne b} n_a n_b .
$$
For a group pair $(a,b)$, broadcast the score vectors as `s_a[:, None]` and
`s_b[None, :]`.

### 2.2. Equalized Odds — TPR and FPR disparities (use $\rho = 10^2$)
$$
\mathrm{EOdds}(w) = \tfrac12\big[(\widetilde{\max}_\rho\,\mathrm{TPR} - \widetilde{\min}_\rho\,\mathrm{TPR}) + (\widetilde{\max}_\rho\,\mathrm{FPR} - \widetilde{\min}_\rho\,\mathrm{FPR})\big].
$$

### 2.3. Equal Opportunity — TPR disparity only (use $\rho = 10^2$)
$$
\mathrm{EOpp}(w) = \widetilde{\max}_\rho\,\mathrm{TPR} - \widetilde{\min}_\rho\,\mathrm{TPR}.
$$


In [22]:
# --- provided helper functions ----------------------------------------------
def soft_max(v, rho):
    return jax.nn.logsumexp(rho * v) / rho


def soft_min(v, rho):
    return -jax.nn.logsumexp(-rho * v) / rho


def group_scores(w, X_list):
    """Per-group score vectors s = X @ w."""
    return [X @ w for X in X_list]


def soft_rates(w, X_list, y_list, rho):
    """Per-group soft TPR and FPR, using sigma(rho * s)."""
    tpr, fpr = [], []
    for X, y in zip(X_list, y_list):
        p = jax.nn.sigmoid(rho * (X @ w))
        tpr.append(jnp.sum(p * (y == 1)) / (jnp.sum(y == 1) + 1e-15))
        fpr.append(jnp.sum(p * (y == 0)) / (jnp.sum(y == 0) + 1e-15))
    return jnp.stack(tpr), jnp.stack(fpr)


In [23]:
def individual_fairness(w, X_list, y_list):
    """Similarity-weighted cross-group squared score gap (lower = fairer)."""
    s = group_scores(w, X_list)
    S = len(X_list)
    Z = sum(len(y_list[a]) * len(y_list[b])
            for a in range(S) for b in range(S) if a != b)
    total = 0.0
    for a in range(S):
        for b in range(a + 1, S):
            sim = jnp.exp(-jnp.abs(y_list[a][:, None] - y_list[b][None, :]))
            gap = (s[a][:, None] - s[b][None, :]) ** 2
            total = total + jnp.sum(sim * gap)
    return total / Z


def equalized_odds(w, X_list, y_list, rho=1e2):
    """Smooth TPR + FPR disparity across groups (use rho = 1e2)."""
    tpr, fpr = soft_rates(w, X_list, y_list, rho)
    return 0.5 * ((soft_max(tpr, rho) - soft_min(tpr, rho))
                  + (soft_max(fpr, rho) - soft_min(fpr, rho)))


def equal_opportunity(w, X_list, y_list, rho=1e2):
    """Smooth TPR disparity across groups (use rho = 1e2)."""
    tpr, _ = soft_rates(w, X_list, y_list, rho)
    return soft_max(tpr, rho) - soft_min(tpr, rho)

### Checkpoint 1: Compare Against the Toolbox

Fit the model with equal group weights $w^\star(\tfrac13, \tfrac13, \tfrac13)$
and compare your metric values with `badr.metrics`. The differences should be
near numerical precision. If a check fails, first inspect the normalization,
whether group pairs are counted once or twice, and whether you used soft rather
than thresholded predictions.


In [24]:
METRICS = {
    "Individual Fairness": individual_fairness,
    "Equalized Odds":      equalized_odds,
    "Equal Opportunity":   equal_opportunity,
}

w_erm = lh.fit_weights(model, dset, lh.uniform_weights(dset))
lh.check_close("Individual Fairness", individual_fairness(w_erm, X_list, y_list),
               badr.metrics.IndividualFairness().fun(w_erm, dset))
lh.check_close("Equalized Odds", equalized_odds(w_erm, X_list, y_list),
               badr.metrics.EqualizedOdds(rho=1e2).fun(w_erm, dset))
lh.check_close("Equal Opportunity", equal_opportunity(w_erm, X_list, y_list),
               badr.metrics.EqualOpportunity(rho=1e2).fun(w_erm, dset))

[PASS] Individual Fairness    yours=0.025257  ref=0.025257  max|delta|=0.00e+00
[PASS] Equalized Odds         yours=0.241311  ref=0.241311  max|delta|=0.00e+00
[PASS] Equal Opportunity      yours=0.13221  ref=0.13221  max|delta|=8.33e-17


True

## 3. Fixed Group Weightings <a class="anchor" id="sec-reweight"></a>

Before using BADR to choose $\lambda$, evaluate a few fixed choices. These are
provided by the helper module or the toolbox.

- **Uniform** — $\lambda_s = 1/S$ (plain ERM); `lh.uniform_weights`.
- **Balanced by group size** — inverse group size, $\lambda_s \propto 1/n_s$;
  `lh.balanced_weights`.
- **Min-max** — minimize the largest group loss, $\min_w \max_s \ell_s(w)$. The
  normalized dual weights are returned by `lh.minmax_reference`, which calls the
  toolbox's `NonsmoothMinMaxLogisticRegression`.
- **Vertices** $e_1, e_2, e_3$ — put all weight on one group. The best of these
  three (**best single group**) is also a baseline for the simplex plots.

`lh.print_weighting_table` refits $w^\star(\lambda)$ for each weighting and
reports the three fairness metrics and the largest group loss $\max_s \ell_s(w)$.


In [25]:
_, lam_minmax = lh.minmax_reference(dset, l2_reg=1e0)

weightings = {
    "Uniform":       lh.uniform_weights(dset),
    "Balanced":      lh.balanced_weights(dset),
    "Min-max":       lam_minmax,
    "e1 (group 1)":  jnp.eye(n_groups)[0],
    "e2 (group 2)":  jnp.eye(n_groups)[1],
    "e3 (group 3)":  jnp.eye(n_groups)[2],
}
lh.print_weighting_table(model, dset, METRICS, weightings)


   weighting  Individual Fairness    Equalized Odds  Equal Opportunity  worst-group loss
------------  -------------------  ----------------  -----------------  ----------------
     Uniform               0.0253            0.2413             0.1322            0.6827
    Balanced               0.0234            0.1863             0.0905            0.6808
     Min-max               0.0243            0.2452             0.1489            0.6762
e1 (group 1)               0.0423            0.3282             0.3169            0.7186
e2 (group 2)               0.0976            0.3012             0.5601            0.7878
e3 (group 3)               0.1307            0.7752             0.7572            0.7874


**Read the table.** Does the best weighting for one fairness metric also give
the best value for the others? Does size balancing lower the fairness metrics, or
does it mainly change the largest group loss? What happens when all weight is put
on one group? These comparisons motivate choosing $\lambda$ for the metric of
interest rather than relying on one fixed rule.


## 4. Run BADR <a class="anchor" id="sec-badr"></a>

Now use the high-level interface. Given a dataset, a model, and a metric,
`badr.Badr` chooses group weights, refits the model, and stores the learned
weights, coefficients, and metric value.


In [ ]:
# Problem definition.
metric = badr.metrics.IndividualFairness()
metric.set_model(model)

# Run BADR for this metric.
b = badr.Badr(dset, model, metric)
b.run()

badr_fairness = metric.fun(b.coef_, dset)

print("BADR group weights:", np.round(np.asarray(b.group_weights), 3))
print("fairness (IF):     ", round(float(badr_fairness), 4))


Compare BADR's learned weights and individual-fairness value with the fixed
weightings in the table. Is the BADR value lower than the values from uniform,
size-balanced, min-max, and the best single-group weighting?


## 5. Simplex Plots <a class="anchor" id="sec-figure"></a>

The last experiment samples the three-group simplex. For each fairness metric,
the color shows $V(\lambda) = \mathrm{metric}(w^\star(\lambda))$; lighter colors
mean smaller metric values. The markers show four reference weightings:

- **Uniform**, **Balanced**, and **Min-max** from the previous section, and
- **BADR**, the weighting learned for that metric.

Refitting the model across the grid may take a few minutes.

In [ ]:
panels = []
for name, metric in [
    ("Individual Fairness", badr.metrics.IndividualFairness()),
    ("Equalized Odds",      badr.metrics.EqualizedOdds(rho=1e2)),
    ("Equal Opportunity",   badr.metrics.EqualOpportunity(rho=1e2)),
]:
    metric.set_model(model)
    print(f"[{name}] running BADR ...")
    b = badr.Badr(dset, model, metric)
    b.run()
    markers = lh.default_markers({
        "Uniform":     lh.uniform_weights(dset),
        "Balanced":    lh.balanced_weights(dset),
        "Min-max":     lam_minmax,
        "BADR":        np.asarray(b.group_weights),
    })
    panels.append((name, METRICS[name], markers))

fig = lh.plot_simplex_figure(
    dset, model, panels, grid_n=20
)

## 6. Discussion <a class="anchor" id="sec-discussion"></a>

Use the table and simplex plots to answer:

1. For each metric, is BADR in a region with smaller values than Uniform?
2. Does Min-max help all fairness metrics, or only some of them?
3. Between Uniform and Balanced, which is more competitive here? What does that
   suggest about the role of group sizes in this dataset?
4. Do the BADR weights for the three metrics lie in the same part of the simplex?
   What does this say about optimizing one metric and reporting another?

### Further Checks

- Try another metric: `badr.metrics.DemographicParity`, `GroupVariance`, `HSIC`,
  or `DisparateMistreatment`.
- Try another dataset: `fetch_adult`, `fetch_compas`, or `fetch_lawschool`.
- Try BADR's stochastic option: `badr.Badr(dset, model, metric, oracle="stochastic")`.

<div align="right"><a href="#toc">Back to TOC</a></div>
